# Unsloth Qwen3-1.7B SFT / QLoRA / GRPO 完整实验 Notebook

这个 notebook 是按 Unsloth 官方 notebook 的风格写的，但把仓库里脚本已有的常用功能都集中到一个交互式工作台里：

- 从你本地的 `model/Qwen3-1.7B` 加载模型。
- 4-bit QLoRA / LoRA SFT。
- response-only loss（默认，只训练 assistant answer）或 full prompt+response loss。
- 保存 LoRA adapter，或导出 merged 16-bit / merged 4-bit 模型。
- 单条推理。
- JSON / JSONL 批量推理，支持 `prompt` / `messages`、`num_repeats`、输出为 list。
- 可选 GRPO / RL，包含 `contains` / `exact` / `numeric` baseline reward。

> 默认假设你已经把 `Qwen3-1.7B` 放在仓库根目录的 `model/Qwen3-1.7B`。如果目录名不同，只需要改下面配置 cell 的 `MODEL_NAME_OR_PATH`。

## 0. 安装依赖（按需）

推荐先在仓库根目录安装：

```bash
pip install -r requirements.txt
```

如果你是在临时 notebook 环境里运行，可以取消下面 cell 的注释。

In [ ]:
# %%capture
# !pip install -r ../requirements.txt

## 1. 全局配置

这里集中配置模型路径、数据文件、输出目录、LoRA 超参、训练开关和推理参数。Notebook 会自动判断当前工作目录是仓库根目录还是 `notebooks/` 目录。

In [ ]:
from pathlib import Path
import json
import os
import re
import sys
from typing import Any

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

# 你已经下载好的本地模型目录
MODEL_NAME_OR_PATH = str(REPO_ROOT / "model" / "Qwen3-1.7B")

# 数据与输出
TRAIN_FILE = REPO_ROOT / "data" / "toy_sft.jsonl"
PROMPT_FILE = REPO_ROOT / "data" / "prompts.json"
OUTPUT_DIR = REPO_ROOT / "outputs" / "qwen3_1p7b_unsloth_notebook_lora"
BATCH_OUTPUT_FILE = REPO_ROOT / "outputs" / "qwen3_1p7b_notebook_batch_outputs.json"

# 数据字段
PROMPT_FIELD = "prompt"
RESPONSE_FIELD = "groundtruth"
MESSAGES_FIELD = "messages"
SPLIT_FIELD = None
SPLIT = None

# 模型 / LoRA / QLoRA
MAX_SEQ_LENGTH = 1024
DTYPE = None          # None = Unsloth 自动选择；也可设 torch.float16 / torch.bfloat16 / torch.float32
LOAD_IN_4BIT = True   # True = QLoRA 4-bit 加载；False = 普通 LoRA
CHAT_TEMPLATE = None  # 例如 "chatml"；None 表示使用 tokenizer 自带 chat_template
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
RANDOM_STATE = 3407

# SFT 训练
LOSS_ON_PROMPT = False       # False = response-only loss；True = prompt+response 都算 loss
RUN_SFT_TRAIN = True         # 如果只想看流程不训练，改成 False
RESUME_FROM_CHECKPOINT = None
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
SAVE_METHOD = "lora"        # "lora" / "merged_16bit" / "merged_4bit"

# 推理 / batch infer
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.7
TOP_P = 0.9
MIN_P = None
BATCH_SIZE = 4
NUM_REPEATS = 1
ALWAYS_LIST_OUTPUT = False
OUTPUT_FIELD = "output"
SYSTEM_PROMPT = None
RUN_BATCH_INFER = True

# GRPO / RL
RUN_GRPO_TRAIN = False       # 默认不跑 RL；确认 reward 后再改 True
REWARD_TYPE = "contains"    # "contains" / "exact" / "numeric"
GRPO_OUTPUT_DIR = REPO_ROOT / "outputs" / "qwen3_1p7b_unsloth_notebook_grpo"
GRPO_MAX_PROMPT_LENGTH = 768
GRPO_MAX_COMPLETION_LENGTH = 256
GRPO_MAX_STEPS = 100
GRPO_LEARNING_RATE = 5e-6
GRPO_NUM_GENERATIONS = 2
GRPO_BETA = 0.0
GRPO_FAST_INFERENCE = False  # 若配置了 vLLM，可改 True

print("Repo root:", REPO_ROOT)
print("Model path:", MODEL_NAME_OR_PATH)
print("Model path exists:", Path(MODEL_NAME_OR_PATH).exists())
print("Train file:", TRAIN_FILE)
print("Prompt file:", PROMPT_FILE)

## 2. CUDA / 环境检查

In [ ]:
import torch
from llm_lab.model_utils import print_cuda_info

print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print_cuda_info()

if not Path(MODEL_NAME_OR_PATH).exists():
    raise FileNotFoundError(
        f"没有找到本地模型目录：{MODEL_NAME_OR_PATH}\n"
        "请确认你已经把 Qwen3-1.7B 放到仓库根目录的 model/Qwen3-1.7B，"
        "或者修改配置 cell 里的 MODEL_NAME_OR_PATH。"
    )

## 3. 通用工具函数（数据读写、chat 渲染、batch infer、reward）

这部分把 `scripts/batch_infer_lora.py` 和 `scripts/train_grpo.py` 中常用的功能放进 notebook，方便你直接改。

In [ ]:
from llm_lab.data import _apply_chat_template, load_grpo_dataset, load_sft_dataset
from llm_lab.model_utils import ensure_pad_token


def read_records(path: str | Path) -> list[dict[str, Any]]:
    input_path = Path(path)
    raw = input_path.read_text(encoding="utf-8").strip()
    if not raw:
        raise ValueError(f"No rows found in {input_path}")
    if input_path.suffix.lower() == ".json" or raw.startswith("["):
        data = json.loads(raw)
        if not isinstance(data, list):
            raise ValueError(f"JSON input {input_path} must be a list of objects.")
        rows = data
    else:
        rows = []
        for line_no, line in enumerate(raw.splitlines(), start=1):
            stripped = line.strip()
            if not stripped:
                continue
            obj = json.loads(stripped)
            if not isinstance(obj, dict):
                raise ValueError(f"Line {line_no} in {input_path} must be a JSON object.")
            rows.append(obj)
    for idx, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(f"Input item {idx} in {input_path} must be a JSON object.")
    return rows


def infer_output_format(output_path: Path, output_format: str = "auto") -> str:
    if output_format != "auto":
        return output_format
    return "jsonl" if output_path.suffix.lower() == ".jsonl" else "json"


def write_records(path: str | Path, records: list[dict[str, Any]], output_format: str = "auto") -> None:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fmt = infer_output_format(output_path, output_format)
    with output_path.open("w", encoding="utf-8") as handle:
        if fmt == "jsonl":
            for item in records:
                handle.write(json.dumps(item, ensure_ascii=False) + "\n")
        else:
            json.dump(records, handle, ensure_ascii=False, indent=2)
            handle.write("\n")


def row_to_messages(row: dict[str, Any], system_prompt: str | None = None) -> list[dict[str, str]]:
    if MESSAGES_FIELD in row:
        messages = row[MESSAGES_FIELD]
        if not isinstance(messages, list):
            raise ValueError(f"'{MESSAGES_FIELD}' must be a list of chat messages.")
        return messages
    prompt = row.get(PROMPT_FIELD)
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError(f"Each row must contain non-empty '{PROMPT_FIELD}' or '{MESSAGES_FIELD}'.")
    messages: list[dict[str, str]] = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    return messages


def build_work_items(texts: list[str], num_repeats: int) -> list[tuple[int, str]]:
    if num_repeats < 1:
        raise ValueError("num_repeats must be >= 1")
    return [(row_idx, text) for row_idx, text in enumerate(texts) for _ in range(num_repeats)]


def batched(items: list[Any], size: int):
    for start in range(0, len(items), size):
        yield start, items[start : start + size]


def normalize_text(value: str) -> str:
    return " ".join(value.casefold().strip().split())


def last_number(value: str) -> str | None:
    matches = re.findall(r"[-+]?\d+(?:\.\d+)?", value.replace(",", ""))
    return matches[-1] if matches else None


def completion_to_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        return "\n".join(
            item.get("content", str(item)) if isinstance(item, dict) else str(item)
            for item in completion
        )
    return str(completion)


def make_reward_func(reward_type: str):
    def reward_func(completions, answer, **_: Any) -> list[float]:
        rewards: list[float] = []
        for completion, expected in zip(completions, answer):
            generated = completion_to_text(completion)
            if reward_type == "exact":
                rewards.append(1.0 if normalize_text(generated) == normalize_text(expected) else 0.0)
            elif reward_type == "numeric":
                rewards.append(1.0 if last_number(generated) == last_number(expected) and last_number(expected) else 0.0)
            else:
                rewards.append(1.0 if normalize_text(expected) in normalize_text(generated) else 0.0)
        return rewards
    return reward_func

## 4. 加载本地 Qwen3-1.7B，并注入 LoRA

这就是 Unsloth 官方参考代码的核心结构：`FastLanguageModel.from_pretrained(...)` + `FastLanguageModel.get_peft_model(...)`。

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME_OR_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

if CHAT_TEMPLATE:
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)

ensure_pad_token(tokenizer)
tokenizer.padding_side = "left"

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = RANDOM_STATE,
    use_rslora = False,
    loftq_config = None,
)

## 5. 加载 SFT 数据

- `LOSS_ON_PROMPT = False`：response-only loss，和 `scripts/train_lora.py` 默认行为一致。
- `LOSS_ON_PROMPT = True`：用 TRL `SFTTrainer` 训练完整 prompt+response。

In [ ]:
train_dataset = load_sft_dataset(
    TRAIN_FILE,
    tokenizer,
    prompt_field = PROMPT_FIELD,
    response_field = RESPONSE_FIELD,
    split_field = SPLIT_FIELD,
    split = SPLIT,
    response_only_loss = not LOSS_ON_PROMPT,
)
print(train_dataset)
print(train_dataset[0].keys())

## 6. 构建 SFT Trainer

为了把脚本功能完整带进 notebook，这里同时支持：

- response-only loss：`transformers.Trainer` + `ResponseOnlyDataCollator`。
- full loss：TRL `SFTTrainer`。

In [ ]:
from transformers import Trainer
from llm_lab.train_utils import ResponseOnlyDataCollator, build_sft_trainer, get_training_args

fp16 = not is_bfloat16_supported()
bf16 = is_bfloat16_supported()

training_args = get_training_args(
    output_dir = str(OUTPUT_DIR),
    max_length = MAX_SEQ_LENGTH,
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    fp16 = fp16,
    bf16 = bf16,
)

if LOSS_ON_PROMPT:
    trainer = build_sft_trainer(model, tokenizer, train_dataset, training_args)
else:
    trainer = Trainer(
        model = model,
        tokenizer = tokenizer,
        args = training_args,
        train_dataset = train_dataset,
        data_collator = ResponseOnlyDataCollator(tokenizer, max_length=MAX_SEQ_LENGTH),
    )

trainer

## 7. 运行 SFT，并保存 adapter / merged 模型

In [ ]:
if RUN_SFT_TRAIN:
    trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
else:
    print("RUN_SFT_TRAIN=False，跳过 SFT 训练。")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_METHOD == "lora":
    trainer.model.save_pretrained(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
else:
    trainer.model.save_pretrained_merged(str(OUTPUT_DIR), tokenizer, save_method=SAVE_METHOD)
print(f"Saved to {OUTPUT_DIR} with save_method={SAVE_METHOD}")

## 8. 单条推理

In [ ]:
FastLanguageModel.for_inference(trainer.model)


def generate_one(prompt: str, system_prompt: str | None = SYSTEM_PROMPT) -> str:
    messages: list[dict[str, str]] = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    text = _apply_chat_template(tokenizer, messages, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {key: value.to("cuda") for key, value in inputs.items()}
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }
    if MIN_P is not None:
        generation_kwargs["min_p"] = MIN_P
    outputs = trainer.model.generate(**inputs, **generation_kwargs)
    generated = outputs[:, inputs["input_ids"].shape[-1] :]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()

print(generate_one("请用一句话解释什么是大语言模型。"))

## 9. 批量推理（JSON / JSONL）

这个 cell 对应 `scripts/batch_infer_lora.py` 的核心功能：

- 输入可以是 JSON array 或 JSONL。
- 每条数据可以有 `prompt` 字段，也可以有 `messages` 字段。
- 支持 `NUM_REPEATS` 重复生成。
- `NUM_REPEATS > 1` 时自动把输出保存为 list。

In [ ]:
def batch_generate_records(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    texts = [
        _apply_chat_template(
            tokenizer,
            row_to_messages(row, system_prompt=SYSTEM_PROMPT),
            add_generation_prompt=True,
        )
        for row in rows
    ]
    work_items = build_work_items(texts, NUM_REPEATS)
    outputs_by_row: list[list[str]] = [[] for _ in rows]
    do_sample = TEMPERATURE > 0

    for start, chunk in batched(work_items, BATCH_SIZE):
        chunk_row_indices = [row_idx for row_idx, _ in chunk]
        chunk_texts = [text for _, text in chunk]
        inputs = tokenizer(chunk_texts, return_tensors="pt", padding=True)
        if torch.cuda.is_available():
            inputs = {key: value.to("cuda") for key, value in inputs.items()}
        generation_kwargs = {
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": do_sample,
            "use_cache": True,
            "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        }
        if do_sample:
            generation_kwargs.update({"temperature": TEMPERATURE, "top_p": TOP_P})
            if MIN_P is not None:
                generation_kwargs["min_p"] = MIN_P
        with torch.inference_mode():
            generated = trainer.model.generate(**inputs, **generation_kwargs)
        new_token_ids = generated[:, inputs["input_ids"].shape[-1] :]
        responses = tokenizer.batch_decode(new_token_ids, skip_special_tokens=True)
        for row_idx, response in zip(chunk_row_indices, responses):
            outputs_by_row[row_idx].append(response.strip())
        print(f"Processed generations {min(start + len(chunk), len(work_items))}/{len(work_items)}")

    results: list[dict[str, Any]] = []
    for row, row_outputs in zip(rows, outputs_by_row):
        item = dict(row)
        item[OUTPUT_FIELD] = row_outputs if NUM_REPEATS > 1 or ALWAYS_LIST_OUTPUT else row_outputs[0]
        results.append(item)
    return results


if RUN_BATCH_INFER:
    prompt_rows = read_records(PROMPT_FILE)
    batch_results = batch_generate_records(prompt_rows)
    write_records(BATCH_OUTPUT_FILE, batch_results)
    print(f"Wrote {len(batch_results)} rows to {BATCH_OUTPUT_FILE}")
else:
    print("RUN_BATCH_INFER=False，跳过批量推理。")

## 10. 可选：从保存好的 adapter 重新加载再推理

如果你重启了 kernel，或者只想做推理，可以从 `OUTPUT_DIR` 直接加载保存的 LoRA adapter。这个逻辑对应 `scripts/infer_lora.py` / `scripts/batch_infer_lora.py`。

In [ ]:
# 需要时取消注释：
# infer_model, infer_tokenizer = FastLanguageModel.from_pretrained(
#     model_name = str(OUTPUT_DIR),
#     max_seq_length = MAX_SEQ_LENGTH,
#     dtype = DTYPE,
#     load_in_4bit = LOAD_IN_4BIT,
# )
# ensure_pad_token(infer_tokenizer)
# FastLanguageModel.for_inference(infer_model)

## 11. 可选：GRPO / RL 训练

这是 `scripts/train_grpo.py` 的 notebook 版本。默认 `RUN_GRPO_TRAIN=False`，你确认 reward 逻辑后再打开。实际实验建议把 baseline reward 改成你的任务 reward。

In [ ]:
from unsloth import PatchFastRL
PatchFastRL("grpo", FastLanguageModel)

from trl import GRPOConfig, GRPOTrainer

rl_dataset = load_grpo_dataset(
    TRAIN_FILE,
    prompt_field = PROMPT_FIELD,
    answer_field = RESPONSE_FIELD,
    split_field = SPLIT_FIELD,
    split = SPLIT,
)

reward_func = make_reward_func(REWARD_TYPE)

grpo_kwargs = {
    "output_dir": str(GRPO_OUTPUT_DIR),
    "learning_rate": GRPO_LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": 1,
    "num_generations": GRPO_NUM_GENERATIONS,
    "max_prompt_length": GRPO_MAX_PROMPT_LENGTH,
    "max_completion_length": GRPO_MAX_COMPLETION_LENGTH,
    "max_steps": GRPO_MAX_STEPS,
    "temperature": 1.0,
    "beta": GRPO_BETA,
    "fp16": fp16,
    "bf16": bf16,
    "logging_steps": 1,
    "save_steps": 50,
    "report_to": "none",
}
if GRPO_FAST_INFERENCE:
    grpo_kwargs["use_vllm"] = True

grp_args = GRPOConfig(**grpo_kwargs)
grp_trainer = GRPOTrainer(
    model = trainer.model,
    processing_class = tokenizer,
    reward_funcs = [reward_func],
    args = grp_args,
    train_dataset = rl_dataset,
)

if RUN_GRPO_TRAIN:
    grp_trainer.train()
    grp_trainer.save_model(str(GRPO_OUTPUT_DIR))
    tokenizer.save_pretrained(str(GRPO_OUTPUT_DIR))
    print(f"Saved GRPO adapter to {GRPO_OUTPUT_DIR}")
else:
    print("RUN_GRPO_TRAIN=False，已构建 GRPOTrainer，但未开始训练。")

## 12. 对应的命令行脚本

如果 notebook 流程跑通，后续大规模实验建议用脚本跑：

```bash
CUDA_VISIBLE_DEVICES=0 python scripts/train_lora.py \
  --model_name_or_path model/Qwen3-1.7B \
  --train_file data/toy_sft.jsonl \
  --output_dir outputs/qwen3_1p7b_unsloth_lora

CUDA_VISIBLE_DEVICES=0 python scripts/batch_infer_lora.py \
  --model_name_or_path outputs/qwen3_1p7b_unsloth_lora \
  --input_file data/prompts.json \
  --output_file outputs/qwen3_1p7b_unsloth_batch_outputs.json \
  --overwrite

CUDA_VISIBLE_DEVICES=0 python scripts/train_grpo.py \
  --model_name_or_path model/Qwen3-1.7B \
  --train_file data/toy_sft.jsonl \
  --output_dir outputs/qwen3_1p7b_unsloth_grpo
```